# Deploy and Operate an AI Service

Exercise a versioned service, fallback, canary, telemetry, and rollback in a local deployment.

Run the lecture cells in sequence with Python 3.10 or later. The default exercises use bundled data and require no model account.


## Practice case · When a successful response is really a fallback

Helena operates a review assistant for a utility. A canary route starts timing out. Callers still get responses from a fallback, so the dashboard looks healthy until staff report that manual review queues are growing.

The workplace scenario is fictional.


## Package and observe


### Lecture 01 · Serve a versioned endpoint

Expose a local HTTP endpoint with explicit version and health behavior.

#### Probe the route the caller will actually use

The service reports ready. A smoke request through the public interface gets the wrong version because routing wasn’t updated.

Probe readiness and the actual versioned response separately. Record caller-visible status and active version. It is possible for every component to be up and for the customer-facing route to be wrong.

#### How it works

A model service has an externally testable contract and versioned implementation. Readiness answers whether the service may receive traffic. A smoke test sends a known request across the same transport the caller uses and checks the response and version.

##### Probe the interface the caller uses

A deployable endpoint has a defined request shape, response shape, and behavior when an input is missing or a provider is unavailable. Include served version or another traceable revision, particularly when traffic is split. A health check answers whether a service can receive traffic. An end-to-end smoke request checks which behavior a real caller receives.

Separate process liveness from readiness. A process may be alive while the dependency needed to handle requests has not loaded. Conversely, a fallback could serve a reduced response even though the preferred route is temporarily unavailable. Choose a contract that makes this visible where it matters.

Use a known request in the workbook to verify both the route and its answer or handoff. Test invalid input as well. Capture the active version and actual result so later operators can tell whether traffic returned to the stable implementation.

#### Worked example

A policy helper returns an answer or handoff plus the requested and served versions. A separate health endpoint lets routing detect an unavailable process.

#### Try it

Run the in-process application smoke check. Use the loopback HTTP smoke client in the final lecture to test the same contract through the network stack.


In [ ]:
import json
from time import perf_counter
from uuid import uuid4
from wsgiref.util import setup_testing_defaults


def answer(version, question):
    if version == "v2" and question == "special request":
        raise TimeoutError("candidate timed out")
    return "Use the reservation form." if question == "reserve a room" else "Send to the service desk."


class Service:
    def __init__(self, store=None):
        self.store = store

    def __call__(self, environ, start_response):
        path = environ["PATH_INFO"]
        if path == "/health":
            status, result = "200 OK", {"ready": True}
        elif path == "/predict" and environ["REQUEST_METHOD"] == "POST":
            status, result = self.predict(environ)
        else:
            status, result = "404 Not Found", {"error": "unknown endpoint"}
        body = json.dumps(result).encode()
        start_response(status, [("Content-Type", "application/json"), ("Content-Length", str(len(body)))])
        return [body]

    def predict(self, environ):
        try:
            size = int(environ.get("CONTENT_LENGTH") or "0")
            if not 0 < size <= 4096:
                raise ValueError("invalid length")
            payload = json.loads(environ["wsgi.input"].read(size))
            question = payload.get("question") if isinstance(payload, dict) else None
            if not isinstance(question, str) or not question.strip():
                raise ValueError("missing question")
        except (ValueError, UnicodeError):
            return "400 Bad Request", {"error": "invalid question"}

        canary = environ.get("HTTP_X_AIEBOK_CANARY") == "1"
        requested = self.store.active_version(canary=canary) if self.store else "v1"
        served = requested
        start = perf_counter()
        try:
            result = answer(served, question)
        except TimeoutError:
            served = "v1"
            result = answer(served, question)
        request_id = uuid4().hex
        if self.store:
            self.store.record(request_id, requested, served, (perf_counter() - start) * 1000)
        return "200 OK", {"id": request_id, "requested_version": requested, "served_version": served, "answer": result}


def request(app, path, data=None, *, canary=False):
    from io import BytesIO

    environ = {}
    setup_testing_defaults(environ)
    environ["PATH_INFO"] = path
    if canary:
        environ["HTTP_X_AIEBOK_CANARY"] = "1"
    if data is not None:
        body = json.dumps(data).encode()
        environ.update({"REQUEST_METHOD": "POST", "CONTENT_LENGTH": str(len(body)), "wsgi.input": BytesIO(body)})
    status = []
    payload = b"".join(app(environ, lambda value, headers: status.append(value)))
    return status[0], json.loads(payload)


assert request(Service(), "/health") == ("200 OK", {"ready": True})
status, result = request(Service(), "/predict", {"question": "reserve a room"})
assert status == "200 OK" and result["served_version"] == "v1"
assert request(Service(), "/predict", {"question": ""})[0] == "400 Bad Request"


**Compare your result**

The response identifies the active version and health succeeds without invoking a model.


### Lecture 02 · Fallback, trace fields and durable telemetry

Record provider failures and successful fallback without storing request text.

#### Tell the operator what the fallback did

The fallback returns a result, but Helena sees only a successful transport status. The primary is still failing and work handed to staff is piling up.

Record requested and served versions, elapsed time, and fallback reason without storing sensitive input. Query failures by requested version so a successful fallback doesn’t hide a broken candidate.

#### How it works

A fallback may preserve availability while hiding primary failures from aggregate success. Measure the primary failure rate separately. Capture request ID, requested and served versions, elapsed time, and the fallback outcome. Persist the telemetry so the release gate can query it.

##### Give fallback behavior its own operational record

When a primary route fails, fallback can maintain a useful outcome. It can also hide reliability problems if the dashboard counts every response the same way. A successful handoff might still consume staff time. Record what was requested, what actually served, and why those choices differed.

A structured trace can include a request identifier, attempted version, served version, status, elapsed time, and fallback reason. Avoid storing personal request text where counts and identifiers suffice. Choose retention and access deliberately so support can investigate an incident without making routine observability a new source of exposure.

Persist the events needed for release evaluation. A worker restart should not erase the recent evidence for a failing canary. Run one forced timeout and query the resulting record. If the public response was correct and the events still show a primary failure, the telemetry is describing the system that operators need to run.

#### Worked example

A candidate fails to answer during a dry run. The baseline serves it and the trace preserves the fact that a fallback occurred.

#### Try it

Trigger the bundled provider failure. Query the telemetry and check that sensitive request text is absent.


In [ ]:
import sqlite3
from pathlib import Path
from tempfile import TemporaryDirectory


class Store:
    def __init__(self, filename):
        self.conn = sqlite3.connect(filename)
        with self.conn:
            self.conn.executescript("""
                CREATE TABLE IF NOT EXISTS config (id INTEGER PRIMARY KEY, version TEXT NOT NULL, canary TEXT);
                INSERT OR IGNORE INTO config VALUES (1, 'v1', NULL);
                CREATE TABLE IF NOT EXISTS traces (
                    id TEXT PRIMARY KEY, requested TEXT, served TEXT, elapsed_ms REAL, fallback INTEGER
                );
                CREATE TABLE IF NOT EXISTS decisions (version TEXT, reason TEXT);
            """)

    def active_version(self, *, canary=False):
        version, candidate = self.conn.execute("SELECT version, canary FROM config WHERE id = 1").fetchone()
        return candidate if canary and candidate else version

    def record(self, request_id, requested, served, elapsed_ms):
        with self.conn:
            self.conn.execute("INSERT INTO traces VALUES (?, ?, ?, ?, ?)",
                              (request_id, requested, served, elapsed_ms, int(served != requested)))

    def activate(self, version, reason):
        if version not in {"v1", "v2"}:
            raise ValueError("unknown version")
        with self.conn:
            self.conn.execute("UPDATE config SET version = ?, canary = NULL WHERE id = 1", (version,))
            self.conn.execute("INSERT INTO decisions VALUES (?, ?)", (version, reason))

    def stage(self, version, reason):
        if version not in {"v1", "v2"}:
            raise ValueError("unknown version")
        with self.conn:
            self.conn.execute("UPDATE config SET canary = ? WHERE id = 1", (version,))
            self.conn.execute("INSERT INTO decisions VALUES (?, ?)", (version, reason))


directory = TemporaryDirectory(prefix="aiebok-service-")
database = Path(directory.name) / "release.sqlite"
store = Store(database)
store.stage("v2", "start synthetic canary")
service = Service(store)
status, stable = request(service, "/predict", {"question": "special request"})
assert status == "200 OK" and stable["requested_version"] == "v1"
status, response = request(service, "/predict", {"question": "special request"}, canary=True)
assert status == "200 OK" and response["served_version"] == "v1"
trace = store.conn.execute("SELECT requested, served, fallback FROM traces WHERE id = ?", (response["id"],)).fetchone()
assert trace == ("v2", "v1", 1)
assert "question" not in {column[1] for column in store.conn.execute("PRAGMA table_info(traces)")}


**Compare your result**

The stored row should show the candidate as requested and baseline as served with a fallback flag.


## Release and recover


### Lecture 03 · Canary evaluation and rollout decisions

Compare gated evidence and identify a canary that must pause.

#### Hold the canary even if the aggregate is green

Stable requests dominate traffic. The candidate exceeds its fallback rate for one cohort, but overall success still looks good.

Apply the gate to the candidate cohort with a predeclared denominator. Require enough observations and a tolerable operational burden. Record why you held rollout and the evidence needed to retry.

#### How it works

A canary is a limited rollout of a labeled version. Define sample requirements, acceptable fallback or error rates, and a latency bound. Evaluate on traffic attributed to the requested version, so fallbacks do not improve the apparent reliability of the failing version.

##### Evaluate the candidate cohort on its own merits

A canary limits the group receiving a proposed version. The gate must still examine the candidate group separately. Stable traffic can dominate the aggregate and obscure a regression in a small trial. Record how cases are assigned and which version the request attempted to use.

Choose a minimum useful observation count and high-priority criteria before looking at results. Check latency, fallback reason, and valid user outcome. An aggregate availability signal is useful for alerting, but insufficient for deciding whether a version should receive additional traffic. Compare like workloads so a batch of unusually easy requests does not accidentally justify expansion.

If the candidate fails, preserve both the decision and evidence. Decide whether to hold at the current scope, disable it, or investigate a confounding factor. A good release tool distinguishes “failed the gate” from “insufficient evidence” so operators know whether to change the system or gather more observations.

#### Worked example

If the candidate has two failures in ten comparable requests and the maximum is one in ten, pause even when all ten callers received some response.

#### Try it

Run the gate with a limited candidate cohort and compare its decision with normal traffic. The notebook uses a loopback request header to select the fixture cohort. In a deployed service, assign cohorts in trusted routing infrastructure.


In [ ]:
def gate(store, version, minimum=4, max_fallback_rate=0.2, p95_limit_ms=1000):
    rows = store.conn.execute(
        "SELECT elapsed_ms, fallback FROM traces WHERE requested = ? ORDER BY elapsed_ms", (version,)
    ).fetchall()
    if len(rows) < minimum:
        return False, f"insufficient sample ({len(rows)}/{minimum})"
    rate = sum(fallback for _, fallback in rows) / len(rows)
    from math import ceil
    p95 = rows[ceil(0.95 * len(rows)) - 1][0]
    return rate <= max_fallback_rate and p95 <= p95_limit_ms, f"fallback rate {rate:.1%}, p95 {p95:.2f} ms"


assert gate(store, "v2")[1].startswith("insufficient sample")
for question in ("special request", "reserve a room", "reserve a room"):
    request(service, "/predict", {"question": question}, canary=True)
passed, reason = gate(store, "v2")
assert not passed and "50.0%" in reason
for _ in range(3):
    request(service, "/predict", {"question": "reserve a room"})
assert gate(store, "v1")[0]
print("Canary", reason)


**Compare your result**

An insufficient sample should fail with a coverage reason. A well-sampled unhealthy candidate should fail with the measured rate.


### Lecture 04 · Rollback and production handoff

Persist a version-pointer rollback and verify it through a local HTTP smoke test.

#### Recovery is a new observation

The on-call operator moves the active pointer back to the last stable revision. Staff still need confirmation that new requests use the intended version.

Reopen the state if relevant and send a smoke request through the serving interface. Compare requested and served versions and log the decision. A successful configuration write is not itself evidence of recovery.

#### How it works

Rollback restores the previous serving version. Keep the decision durable and include it in the operator log. After changing the pointer, send a smoke request through the serving endpoint. In hosted environments, use a managed secret store, readiness checks, and an explicit release owner.

##### Verify the service after restoring the pointer

Rollback is a new deployment decision. Use the prior known-good version, record who authorized it and why, then verify the observed route. A successful configuration write can coexist with stale workers, cached policy, or new requests still being routed to the candidate.

Run a smoke check from the perspective of both a stable caller and a caller affected by the candidate. Check the served version, result, and health of any manual queue that received fallback work. If the candidate caused inconsistent or pending effects, decide how operators will reconcile those separately.

Persist the pointer and event history if service continuity depends on surviving restart. Reopen the store in the workbook and repeat the request. The handoff should tell a colleague what failed, which version is safe to use, what remains pending, and which evidence would permit the next release.

#### Worked example

A blocked canary reassigns the production pointer to v1. After reopening its state store, the next request reports v1 and the decision log still contains the rollback.

#### Try it

Run rollback, reopen the store, and make a loopback HTTP health and prediction request. Write your runbook with the measurable trigger and post-rollback check.


In [ ]:
from urllib.request import Request, ProxyHandler, build_opener
from wsgiref.simple_server import make_server, WSGIRequestHandler


class QuietHandler(WSGIRequestHandler):
    def log_message(self, format, *args):
        pass


store.activate("v1", "rollback after blocked canary")
store.conn.close()
reopened = Store(database)
try:
    assert reopened.active_version() == "v1"
    assert reopened.active_version(canary=True) == "v1"
    assert ("v1", "rollback after blocked canary") in reopened.conn.execute("SELECT version, reason FROM decisions").fetchall()
    # The server runs in this thread because the SQLite connection belongs to it.
    from concurrent.futures import ThreadPoolExecutor

    with make_server("127.0.0.1", 0, Service(reopened), handler_class=QuietHandler) as server:
        server.timeout = 5
        address = f"http://127.0.0.1:{server.server_port}"

        def fetch():
            client = build_opener(ProxyHandler({}))
            with client.open(address + "/health", timeout=5) as response:
                assert json.load(response)["ready"]
            call = Request(address + "/predict", data=b'{"question":"reserve a room"}',
                           headers={"Content-Type": "application/json", "X-AIEBOK-CANARY": "1"})
            with client.open(call, timeout=5) as response:
                return json.load(response)

        with ThreadPoolExecutor(max_workers=1) as pool:
            result = pool.submit(fetch)
            server.handle_request()
            server.handle_request()
            assert result.result()["served_version"] == "v1"
finally:
    reopened.conn.close()
    directory.cleanup()
print("Rollback persisted and HTTP smoke passed")


**Compare your result**

The version pointer and decision survive reopen. The deployed service reports its baseline version through HTTP.


## Course project

Ship a local service and a release packet with an architecture decision, version routing, a smoke test, measured canary evidence, a telemetry schema, and an operator runbook.

Demonstrate repeatable smoke requests, fallback under injected provider failure, a blocked canary, and a saved rollback decision. Describe how health checks, secrets, and telemetry would map to a selected hosting platform.
